In [235]:
import pandas as pd
from textblob import TextBlob
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, f1_score, precision_score,
                              recall_score, confusion_matrix,
                              ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import networkx as nx
import re 

# Load the data
df = pd.read_csv('/Users/asfaibrahim/Downloads/final project/stocks.csv')

# NASDAQ's public symbol files
nasdaq = pd.read_csv('https://www.nasdaqtrader.com/dynamic/SymDir/nasdaqlisted.txt', sep='|')
other  = pd.read_csv('https://www.nasdaqtrader.com/dynamic/SymDir/otherlisted.txt',  sep='|')

# Drop last row, metadata footer
nasdaq = nasdaq[:-1]
other  = other[:-1]


print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print(nasdaq.columns.tolist())
print(other.columns.tolist())

Shape: (75857, 24)

Columns:
['id', 'author', 'created', 'retrieved', 'edited', 'pinned', 'archived', 'locked', 'removed', 'deleted', 'is_self', 'is_video', 'is_original_content', 'title', 'link_flair_text', 'upvote_ratio', 'score', 'gilded', 'total_awards_received', 'num_comments', 'num_crossposts', 'selftext', 'thumbnail', 'shortlink']
['Symbol', 'Security Name', 'Market Category', 'Test Issue', 'Financial Status', 'Round Lot Size', 'ETF', 'NextShares']
['ACT Symbol', 'Security Name', 'Exchange', 'CQS Symbol', 'ETF', 'Round Lot Size', 'Test Issue', 'NASDAQ Symbol']


In [236]:
# Look at the first 3 rows
df.head(3)

,id,author,created,retrieved,edited,pinned,archived,locked,removed,deleted,...,link_flair_text,upvote_ratio,score,gilded,total_awards_received,num_comments,num_crossposts,selftext,thumbnail,shortlink
0,ko140x,fonv66,2021-01-01 00:05:17,2021-02-03 21:17:46,1970-01-01 00:00:00,0,0,0,0,0,...,Read the wiki,0.40,0,0,0,9,0,Reposted because I broke rule 4 by mistake. \n...,self,https://redd.it/ko140x
1,ko18qm,Conundrum5,2021-01-01 00:13:13,2021-02-03 21:17:46,1970-01-01 00:00:00,0,0,0,0,0,...,Discussion,0.63,4,0,0,50,0,Also give a rough sense for the size of your p...,self,https://redd.it/ko18qm
2,ko1c6o,[deleted],2021-01-01 00:18:57,2021-02-03 21:17:46,1970-01-01 00:00:00,0,0,0,1,1,...,NaN,0.55,1,0,0,14,0,[deleted],default,https://redd.it/ko1c6o


In [237]:
# Check for missing values
print(df.isnull().sum())

id                           0
author                       0
created                      0
retrieved                    0
edited                       0
pinned                       0
archived                     0
locked                       0
removed                      0
deleted                      0
is_self                      0
is_video                     0
is_original_content          0
title                        0
link_flair_text          35564
upvote_ratio                 0
score                        0
gilded                       0
total_awards_received        0
num_comments                 0
num_crossposts               0
selftext                    47
thumbnail                    0
shortlink                    0
dtype: int64


In [238]:
# Check the data types
print(df.dtypes)

id                        object
author                    object
created                   object
retrieved                 object
edited                    object
pinned                     int64
archived                   int64
locked                     int64
removed                    int64
deleted                    int64
is_self                    int64
is_video                   int64
is_original_content        int64
title                     object
link_flair_text           object
upvote_ratio             float64
score                      int64
gilded                     int64
total_awards_received      int64
num_comments               int64
num_crossposts             int64
selftext                  object
thumbnail                 object
shortlink                 object
dtype: object


In [239]:
# Parse timestamps properly
df['created'] = pd.to_datetime(df['created'])

# Sort by time 
df = df.sort_values('created').reset_index(drop=True)

# Fill missing selftext with empty string
df['selftext'] = df['selftext'].fillna('')

# Remove deleted posts
df = df[df['selftext'] != '[deleted]'].copy()
df = df[df['author'] != '[deleted]'].copy()

# Combine title and selftext into one text field for analysis
df['full_text'] = df['title'] + ' ' + df['selftext']

print(f"Records after cleaning: {len(df)}")
print(f"Date range: {df['created'].min()} to {df['created'].max()}")

Records after cleaning: 66952
Date range: 2021-01-01 00:05:17 to 2021-12-31 22:34:41


In [240]:
# NASDAQ file uses 'Symbol', other file uses 'ACT Symbol'
nasdaq_symbols = set(nasdaq['Symbol'].dropna().str.strip())
other_symbols  = set(other['ACT Symbol'].dropna().str.strip())

valid_symbols = nasdaq_symbols | other_symbols  # union of both sets

print(f"Total known tickers: {len(valid_symbols)}")
# Expect roughly 8,000-10,000 symbols

Total known tickers: 12916


In [241]:
def extract_tickers(text, valid_symbols):
    if not isinstance(text, str):
        return []

    # 1: $TICKER mention = high confidence, keep if real symbol
    dollar_tickers = re.findall(r'\$([A-Z]{1,5})\b', text)

    # 2: bare uppercase words 
    upper_words = re.findall(r'\b([A-Z]{2,5})\b', text)

    # Validate both against the real symbol list
    confirmed = set()

    for t in dollar_tickers:
        if t in valid_symbols:
            confirmed.add(t)

    for t in upper_words:
        if t in valid_symbols:
            confirmed.add(t)

    return list(confirmed)

# Apply it
df['tickers'] = df['full_text'].apply(lambda x: extract_tickers(x, valid_symbols))

In [242]:
# Explode: one row per ticker per post
df_exploded = df_with_tickers.explode('tickers').copy()
df_exploded = df_exploded.rename(columns={'tickers': 'ticker'})

# Remove any empty ticker values
df_exploded = df_exploded[df_exploded['ticker'].notna()].copy()
df_exploded = df_exploded[df_exploded['ticker'] != ''].copy()

# Sort by time again after exploding
df_exploded = df_exploded.sort_values('created').reset_index(drop=True)

print(f"Total rows after exploding: {len(df_exploded)}")
print(f"Unique tickers found: {df_exploded['ticker'].nunique()}")

print("\nTop 20 most mentioned tickers:")
print(df_exploded['ticker'].value_counts().head(20))

Total rows after exploding: 80616
Unique tickers found: 10118

Top 20 most mentioned tickers:
ticker
GME     2152
AMC     1113
AAPL     813
TSLA     743
SPY      742
AMD      601
MSFT     596
QQQ      543
NIO      539
VTI      484
BB       471
PLTR     467
AMZN     464
VOO      445
NVDA     436
BABA     370
TD       365
ARKK     347
WSB      335
FDA      333
Name: count, dtype: int64


In [ ]:
# Keep only posts with at least one valid ticker
df_with_tickers = df[df['tickers'].apply(len) > 0].copy()
df_with_tickers['num_tickers_mentioned'] = df_with_tickers['tickers'].apply(len)

print(f"Posts with tickers: {len(df_with_tickers)}")
print(f"Mean tickers per post: {df_with_tickers['num_tickers_mentioned'].mean():.2f}")

In [ ]:
# Keep only posts with at least one valid ticker
df_with_tickers = df[df['tickers'].apply(len) > 0].copy()
df_with_tickers['num_tickers_mentioned'] = df_with_tickers['tickers'].apply(len)

print(f"Posts with tickers: {len(df_with_tickers)}")
print(f"Mean tickers per post: {df_with_tickers['num_tickers_mentioned'].mean():.2f}")

In [ ]:
# Only keep tickers mentioned at least 10 times total as rare tickers don't have enough data for meaningful surge detection
ticker_counts = df_exploded['ticker'].value_counts()
valid_tickers = ticker_counts[ticker_counts >= 10].index

df_exploded = df_exploded[df_exploded['ticker'].isin(valid_tickers)].copy()
df_exploded = df_exploded.reset_index(drop=True)

print(f"Rows after filtering rare tickers: {len(df_exploded)}")
print(f"Unique tickers remaining: {df_exploded['ticker'].nunique()}")

In [ ]:
def get_sentiment(text):
    """
    Returns a score between -1 (very negative) and +1 (very positive)
    """
    if not isinstance(text, str) or text.strip() == '':
        return 0.0
    return TextBlob(text).sentiment.polarity

print("Computing sentiment scores...")
df_exploded['sentiment_score'] = df_exploded['full_text'].apply(get_sentiment)
print("Done!")

# check progress
print(f"\nSentiment range: {df_exploded['sentiment_score'].min():.3f} to {df_exploded['sentiment_score'].max():.3f}")
print(f"Average sentiment: {df_exploded['sentiment_score'].mean():.3f}")

In [ ]:
# What hour of day was the post made?
df_exploded['hour_of_day'] = df_exploded['created'].dt.hour

# What day of week? (0=Monday, 6=Sunday)
df_exploded['day_of_week'] = df_exploded['created'].dt.dayofweek

# How long is the title?
df_exploded['title_length'] = df_exploded['title'].fillna('').str.split().str.len()

# How long is the body text?
df_exploded['word_count'] = df_exploded['selftext'].fillna('').str.split().str.len()

print("Time and text features done")
print(df_exploded[['hour_of_day', 'day_of_week', 'title_length', 'word_count']].describe())

In [ ]:
# Sort by ticker and time for the rolling calculations
df_exploded = df_exploded.sort_values(['ticker', 'created']).reset_index(drop=True)

print("Computing ticker activity features...")
print("This will take 5-10 minutes. Do not close Jupyter.")

# to store results
post_rate_24h = []
time_since_prev = []
acceleration = []

# Group by ticker and process each ticker's posts together
# This is much faster than processing row by row across all data
grouped = df_exploded.groupby('ticker')

for ticker_name, group in grouped:
    group = group.sort_values('created')
    times = group['created'].tolist()
    
    for i, current_time in enumerate(times):
        
        # --- Feature 1: How many posts about this ticker in prior 24h ---
        count_24h = sum(
            1 for t in times[:i]  # only look at posts BEFORE current
            if (current_time - t).total_seconds() <= 86400  # 86400 seconds = 24 hours
        )
        post_rate_24h.append(count_24h)
        
        # --- Feature 2: How long since the previous post about this ticker ---
        if i == 0:
            # First ever post about this ticker
            time_since_prev.append(24.0)  # assume 24h gap as default
        else:
            gap_hours = (current_time - times[i-1]).total_seconds() / 3600
            time_since_prev.append(min(gap_hours, 72.0))  # cap at 72h
        
        # --- Feature 3: Acceleration (is posting speeding up?) ---
        # Compare last 12h vs the 12h before that
        count_0_12h = sum(
            1 for t in times[:i]
            if (current_time - t).total_seconds() <= 43200  # 12h in seconds
        )
        count_12_24h = sum(
            1 for t in times[:i]
            if 43200 < (current_time - t).total_seconds() <= 86400
        )
        # Ratio > 1 means accelerating, < 1 means slowing down
        accel = count_0_12h / max(count_12_24h, 1)
        acceleration.append(accel)

# Add to dataframe
df_exploded['ticker_post_rate_24h'] = post_rate_24h
df_exploded['time_since_previous'] = time_since_prev
df_exploded['ticker_post_acceleration'] = acceleration

print("Done!")
print(df_exploded[['ticker_post_rate_24h', 'time_since_previous', 'ticker_post_acceleration']].describe())

In [ ]:
print("Computing surge labels...")
print("This will take a few minutes...")

# Sort by time for this calculation
df_exploded = df_exploded.sort_values('created').reset_index(drop=True)

# Store results
volume_growth_list = []

# For each post, look at the FUTURE 24h for the same ticker
for idx, row in df_exploded.iterrows():
    ticker = row['ticker']
    current_time = row['created']
    
    # Get all posts about this ticker
    ticker_posts = df_exploded[df_exploded['ticker'] == ticker]
    
    # Count posts in NEXT 24 hours (future)
    future_posts = ticker_posts[
        (ticker_posts['created'] > current_time) &
        (ticker_posts['created'] <= current_time + pd.Timedelta(hours=24))
    ]
    
    # Count posts in PRIOR 24 hours (past)
    past_posts = ticker_posts[
        (ticker_posts['created'] >= current_time - pd.Timedelta(hours=24)) &
        (ticker_posts['created'] < current_time)
    ]
    
    future_count = len(future_posts)
    past_count = max(len(past_posts), 1)  # avoid division by zero
    
    # Growth rate: how much did activity increase?
    # 1.0 means doubled, 0.0 means stayed same, -0.5 means halved
    growth = (future_count / past_count) - 1
    volume_growth_list.append(growth)
    
    # Print progress every 10000 rows
    if idx % 10000 == 0:
        print(f"  Progress: {idx}/{len(df_exploded)}")

df_exploded['volume_growth'] = volume_growth_list
print("Done!")
print(df_exploded['volume_growth'].describe())

In [ ]:
# Use only training data stats to set the threshold. This s data leakage
split_idx = int(len(df_exploded) * 0.8)

train_mean = df_exploded['volume_growth'].iloc[:split_idx].mean()
train_std = df_exploded['volume_growth'].iloc[:split_idx].std()

print(f"Training mean growth: {train_mean:.3f}")
print(f"Training std growth: {train_std:.3f}")

# Z-score: how many standard deviations above average is this growth?
df_exploded['z_volume'] = (df_exploded['volume_growth'] - train_mean) / train_std

# Label as surge if z-score above threshold
# tau=1.0 means "significantly above average"
tau = 1.0
df_exploded['surge_label'] = (df_exploded['z_volume'] > tau).astype(int)

surge_rate = df_exploded['surge_label'].mean()
print(f"\nSurge rate: {surge_rate:.3f} ({surge_rate*100:.1f}%)")
print(f"Surge posts: {df_exploded['surge_label'].sum()}")
print(f"Non-surge posts: {(df_exploded['surge_label']==0).sum()}")

In [ ]:
# Save the labelled data
df_exploded.to_csv('labelled_data.csv', index=False)
print(f"Saved labelled_data.csv with {len(df_exploded)} rows")

In [ ]:
FEATURES = [
    'hour_of_day',
    'day_of_week', 
    'title_length',
    'word_count',
    'num_tickers_mentioned',
    'sentiment_score',
    'ticker_post_rate_24h',
    'time_since_previous',
    'ticker_post_acceleration'
]

# Drop any rows with missing values in our features
df_model = df_exploded.dropna(subset=FEATURES + ['surge_label']).copy()
df_model = df_model.sort_values('created').reset_index(drop=True)

X = df_model[FEATURES]
y = df_model['surge_label']

# TEMPORAL SPLIT 
# train on the first 80% of tithe me and test on the last 20%
split_idx = int(len(df_model) * 0.8)

X_train = X.iloc[:split_idx]
X_test  = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")
print(f"Train surge rate: {y_train.mean():.3f}")
print(f"Test surge rate:  {y_test.mean():.3f}")

In [ ]:
# GradientBoostingClassifier is sklearn's built-in version of boosting
# Similar to XGBoost but it is easier to implement rightnow due to compatibility issues with XGBoost, which i will attempt to implement in the later stage
models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100,
        random_state=42
    )
}

In [ ]:
FEATURES = [
    'hour_of_day',
    'day_of_week',
    'title_length',
    'word_count',
    'num_tickers_mentioned',
    'sentiment_score',
    'ticker_post_rate_24h',
    'time_since_previous',
    'ticker_post_acceleration'
]

df_model = df_exploded.dropna(subset=FEATURES + ['surge_label']).copy()
df_model = df_model.sort_values('created').reset_index(drop=True)

X = df_model[FEATURES]
y = df_model['surge_label']

# Temporal split, train on first 80%, test on last 20%
split_idx = int(len(df_model) * 0.8)

X_train = X.iloc[:split_idx]
X_test  = X.iloc[split_idx:]
y_train = y.iloc[:split_idx]
y_test  = y.iloc[split_idx:]

print(f"Training samples: {len(X_train)}")
print(f"Test samples:     {len(X_test)}")
print(f"Train surge rate: {y_train.mean():.3f}")
print(f"Test surge rate:  {y_test.mean():.3f}")

# Scale features
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

imbalance_ratio = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Class imbalance ratio: {imbalance_ratio:.1f}x")

In [ ]:
results = []
trained_models = {}

for name, model in models.items():
    print(f"Training {name}...")
    
    model.fit(X_train_sc, y_train)
    
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    
    auc  = roc_auc_score(y_test, y_prob)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    
    results.append({
        'Model':     name,
        'AUC-ROC':   round(auc,  4),
        'F1':        round(f1,   4),
        'Precision': round(prec, 4),
        'Recall':    round(rec,  4),
    })
    
    trained_models[name] = (model, y_prob, y_pred)
    print(f"  AUC-ROC: {auc:.4f}  F1: {f1:.4f}")

print("\n--- Final Results ---")
results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
# ROC Curves
fig, ax = plt.subplots(figsize=(8, 6))

for name, (model, y_prob, y_pred) in trained_models.items():
    RocCurveDisplay.from_predictions(
        y_test, y_prob,
        name=name,
        ax=ax
    )

ax.plot([0, 1], [0, 1], 'k--', label='Random baseline (AUC=0.50)')
ax.set_title('ROC Curves — Surge Prediction Models\nr/stocks 2021', fontsize=13)
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('roc_curves.png', dpi=150)
plt.show()
print("Saved roc_curves.png")

In [ ]:
# Confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, (model, y_prob, y_pred)) in zip(axes, trained_models.items()):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(
        cm,
        display_labels=['No Surge', 'Surge']
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(name, fontsize=11)

plt.suptitle('Confusion Matrices — Surge Prediction', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150)
plt.show()
print("Saved confusion_matrices.png")

In [ ]:
# Feature importance from Random Forest
rf_model = trained_models['Random Forest'][0]
importances = pd.Series(
    rf_model.feature_importances_,
    index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
importances.plot(kind='barh', ax=ax, color='steelblue', alpha=0.85)
ax.set_title('Feature Importance — Random Forest', fontsize=13)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150)
plt.show()
print("Saved feature_importance.png")

In [ ]:
print("Building ticker co-mention network...")

# Sort by time
df_with_tickers_sorted = df_with_tickers.sort_values('created').reset_index(drop=True)

# Only use posts mentioning more than one ticker
multi_ticker_posts = df_with_tickers_sorted[
    df_with_tickers_sorted['num_tickers_mentioned'] > 1
].copy()

print(f"Posts mentioning 2+ tickers: {len(multi_ticker_posts)}")

In [ ]:
# Build the graph
G = nx.Graph()

# For each post mentioning multiple tickers,
# connect those tickers with an edge
for _, row in multi_ticker_posts.iterrows():
    tickers = row['tickers']
    
    # Filter to only valid tickers (ones in our filtered list)
    valid = [t for t in tickers if t in valid_tickers]
    
    # Connect each pair of tickers mentioned together
    for i in range(len(valid)):
        for j in range(i+1, len(valid)):
            a, b = valid[i], valid[j]
            if G.has_edge(a, b):
                G[a][b]['weight'] += 1
            else:
                G.add_edge(a, b, weight=1)

# Add all valid tickers as nodes even if isolated
for ticker in valid_tickers:
    if ticker not in G:
        G.add_node(ticker)

print(f"Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

In [ ]:
# Calculate surge rate per ticker
# This tells us which tickers surged most often
ticker_surge_rate = df_exploded.groupby('ticker')['surge_label'].mean()
ticker_post_count = df_exploded.groupby('ticker').size()

# Add surge info to graph nodes
for node in G.nodes():
    G.nodes[node]['surge_rate'] = ticker_surge_rate.get(node, 0)
    G.nodes[node]['post_count'] = ticker_post_count.get(node, 1)

print("Node attributes added")

In [ ]:
# The full graph with 1326 tickers is too dense to visualise
# Keep only the top 100 most connected tickers
degree = dict(G.degree())
top_100 = sorted(degree.keys(), key=lambda x: degree[x], reverse=True)[:100]
G_sub = G.subgraph(top_100).copy()

print(f"Subgraph: {G_sub.number_of_nodes()} nodes, {G_sub.number_of_edges()} edges")

# Calculate centrality measures on the subgraph
degree_centrality = nx.degree_centrality(G_sub)
betweenness = nx.betweenness_centrality(G_sub)

# Add to nodes
for node in G_sub.nodes():
    G_sub.nodes[node]['degree_centrality'] = degree_centrality[node]
    G_sub.nodes[node]['betweenness'] = betweenness[node]

In [ ]:
# Do more central tickers surge more often? RE: cascade diffusion literature

centrality_df = pd.DataFrame({
    'ticker': list(G_sub.nodes()),
    'degree_centrality': [G_sub.nodes[n]['degree_centrality'] for n in G_sub.nodes()],
    'betweenness': [G_sub.nodes[n]['betweenness'] for n in G_sub.nodes()],
    'surge_rate': [G_sub.nodes[n]['surge_rate'] for n in G_sub.nodes()],
    'post_count': [G_sub.nodes[n]['post_count'] for n in G_sub.nodes()]
})

# Correlation between centrality and surge rate
corr_degree = centrality_df['degree_centrality'].corr(centrality_df['surge_rate'])
corr_between = centrality_df['betweenness'].corr(centrality_df['surge_rate'])

print(f"Correlation: degree centrality vs surge rate: {corr_degree:.3f}")
print(f"Correlation: betweenness centrality vs surge rate: {corr_between:.3f}")

print("\nTop 10 tickers by degree centrality:")
print(centrality_df.nlargest(10, 'degree_centrality')[
    ['ticker', 'degree_centrality', 'surge_rate', 'post_count']
].to_string(index=False))

print("\nTop 10 tickers by surge rate (min 5 posts):")
high_volume = centrality_df[centrality_df['post_count'] >= 5]
print(high_volume.nlargest(10, 'surge_rate')[
    ['ticker', 'surge_rate', 'degree_centrality', 'post_count']
].to_string(index=False))

In [ ]:
pos = nx.spring_layout(G_sub, seed=42, k=0.8)

post_counts = [G_sub.nodes[n]['post_count'] for n in G_sub.nodes()]
max_count = max(post_counts)
node_sizes = [200 + (c / max_count) * 1000 for c in post_counts]
surge_rates = [G_sub.nodes[n]['surge_rate'] for n in G_sub.nodes()]
edge_weights = [G_sub[u][v]['weight'] for u, v in G_sub.edges()]
max_weight = max(edge_weights) if edge_weights else 1
edge_widths = [0.3 + (w / max_weight) * 2 for w in edge_weights]

fig, ax = plt.subplots(figsize=(14, 10))

nx.draw_networkx_edges(
    G_sub, pos, ax=ax,
    width=edge_widths, alpha=0.3, edge_color='grey'
)

nodes = nx.draw_networkx_nodes(
    G_sub, pos, ax=ax,
    node_size=node_sizes,
    node_color=surge_rates,
    cmap=plt.cm.RdYlGn_r,
    alpha=0.85, vmin=0, vmax=0.3
)

# Label top 15 by degree centrality (most mentioned)
top_degree = sorted(degree_centrality.keys(),
                    key=lambda x: degree_centrality[x],
                    reverse=True)[:15]

# ALSO label top 5 by surge rate (the interesting red ones)
top_surge = centrality_df.nlargest(5, 'surge_rate')['ticker'].tolist()

# Combine both label sets
all_labels = set(top_degree + top_surge)
labels = {n: n for n in G_sub.nodes() if n in all_labels}

nx.draw_networkx_labels(
    G_sub, pos, labels=labels,
    ax=ax, font_size=8, font_weight='bold'
)

plt.colorbar(nodes, ax=ax, label='Surge Rate (red = higher)')
ax.set_title(
    'Ticker Co-mention Network — r/stocks 2021\n'
    '(node size = post volume, colour = surge rate, '
    'edge thickness = co-mention frequency)',
    fontsize=12
)
ax.axis('off')
plt.tight_layout()
plt.savefig('network_graph_v2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved network_graph_v2.png")

In [ ]:
# scatter of centrality vs surge rate
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(
    centrality_df['degree_centrality'],
    centrality_df['surge_rate'],
    alpha=0.6,
    color='steelblue',
    s=50
)

# Label a few interesting points
for _, row in centrality_df.nlargest(8, 'degree_centrality').iterrows():
    ax.annotate(
        row['ticker'],
        (row['degree_centrality'], row['surge_rate']),
        fontsize=8,
        xytext=(5, 5),
        textcoords='offset points'
    )

ax.set_xlabel('Degree Centrality')
ax.set_ylabel('Surge Rate')
ax.set_title('Network Centrality vs Surge Rate\nTicker Co-mention Network — r/stocks 2021')
plt.tight_layout()
plt.savefig('centrality_vs_surge.png', dpi=150)
plt.show()
print("Saved centrality_vs_surge.png")

In [ ]:
# Save the labelled dataframe with all features
df_exploded.to_csv('final_data.csv', index=False)

# Save centrality results
centrality_df.to_csv('centrality_results.csv', index=False)

# Save model results
results_df.to_csv('model_results.csv', index=False)

print("All files saved:")
print("  final_data.csv")
print("  centrality_results.csv") 
print("  model_results.csv")
print("  roc_curves.png")
print("  confusion_matrices.png")
print("  feature_importance.png")
print("  network_graph.png")
print("  centrality_vs_surge.png")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# --- Plot 1: ROC Curves ---
ax1 = axes[0, 0]
for name, (model, y_prob, y_pred) in trained_models.items():
    RocCurveDisplay.from_predictions(
        y_test, y_prob, name=name, ax=ax1
    )
ax1.plot([0,1],[0,1],'k--', label='Random (AUC=0.50)')
ax1.set_title('ROC Curves — Surge Prediction')
ax1.legend(fontsize=8)

# --- Plot 2: Feature Importance ---
ax2 = axes[0, 1]
importances = pd.Series(
    trained_models['Random Forest'][0].feature_importances_,
    index=FEATURES
).sort_values(ascending=True)
importances.plot(kind='barh', ax=ax2, color='steelblue', alpha=0.85)
ax2.set_title('Feature Importance — Random Forest')
ax2.set_xlabel('Importance Score')

# --- Plot 3: Surge rate by hour of day ---
ax3 = axes[1, 0]
hourly_surge = df_model.groupby('hour_of_day')['surge_label'].mean()
ax3.bar(hourly_surge.index, hourly_surge.values, color='steelblue', alpha=0.85)
ax3.set_title('Surge Rate by Hour of Day')
ax3.set_xlabel('Hour (UTC)')
ax3.set_ylabel('Surge Rate')
ax3.axhline(
    y=df_model['surge_label'].mean(), 
    color='red', linestyle='--', 
    label=f'Overall avg ({df_model["surge_label"].mean():.3f})'
)
ax3.legend(fontsize=8)

# --- Plot 4: Centrality vs Surge Rate ---
ax4 = axes[1, 1]
ax4.scatter(
    centrality_df['betweenness'],
    centrality_df['surge_rate'],
    alpha=0.6, color='steelblue', s=50
)
ax4.set_title(f'Betweenness Centrality vs Surge Rate\n(r = {corr_between:.3f})')
ax4.set_xlabel('Betweenness Centrality')
ax4.set_ylabel('Surge Rate')

plt.suptitle(
    'Surge Prediction Summary — r/stocks 2021',
    fontsize=14, fontweight='bold', y=1.02
)
plt.tight_layout()
plt.savefig('summary_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved summary_dashboard.png")

In [ ]:
import sys
!{sys.executable} -m pip install imbalanced-learn

In [ ]:
from imblearn.over_sampling import SMOTE

print(f"Before SMOTE:")
print(f"  No surge: {(y_train == 0).sum()}")
print(f"  Surge:    {(y_train == 1).sum()}")

# Apply SMOTE to training data only
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_sc, y_train)

print(f"\nAfter SMOTE:")
print(f"  No surge: {(y_train_sm == 0).sum()}")
print(f"  Surge:    {(y_train_sm == 1).sum()}")

In [ ]:
results_smote = []
trained_models_smote = {}

for name, model in models.items():
    print(f"Training {name} with SMOTE...")
    
    # Train on SMOTE resampled data
    model.fit(X_train_sm, y_train_sm)
    
    # Test on ORIGINAL test set - never apply SMOTE to test data
    y_pred = model.predict(X_test_sc)
    y_prob = model.predict_proba(X_test_sc)[:, 1]
    
    auc  = roc_auc_score(y_test, y_prob)
    f1   = f1_score(y_test, y_pred, zero_division=0)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    
    results_smote.append({
        'Model':     name,
        'AUC-ROC':   round(auc,  4),
        'F1':        round(f1,   4),
        'Precision': round(prec, 4),
        'Recall':    round(rec,  4),
    })
    
    trained_models_smote[name] = (model, y_prob, y_pred)
    print(f"  AUC-ROC: {auc:.4f}  F1: {f1:.4f}  Recall: {rec:.4f}")

print("\n--- SMOTE Results ---")
results_smote_df = pd.DataFrame(results_smote)
print(results_smote_df.to_string(index=False))

In [ ]:
print("--- Before SMOTE ---")
print(results_df.to_string(index=False))
print("\n--- After SMOTE ---")
print(results_smote_df.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, (name, (model, y_prob, y_pred)) in zip(axes, trained_models_smote.items()):
    cm = confusion_matrix(y_test, y_pred)
    ConfusionMatrixDisplay(
        cm,
        display_labels=['No Surge', 'Surge']
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{name}\n(with SMOTE)', fontsize=11)

plt.suptitle('Confusion Matrices — After SMOTE', fontsize=13)
plt.tight_layout()
plt.savefig('confusion_matrices_smote.png', dpi=150)
plt.show()
print("Saved confusion_matrices_smote.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Before SMOTE
ax1 = axes[0]
for name, (model, y_prob, y_pred) in trained_models.items():
    RocCurveDisplay.from_predictions(
        y_test, y_prob, name=name, ax=ax1
    )
ax1.plot([0,1],[0,1],'k--', label='Random baseline')
ax1.set_title('ROC Curves — Before SMOTE')
ax1.legend(fontsize=8)

# After SMOTE
ax2 = axes[1]
for name, (model, y_prob, y_pred) in trained_models_smote.items():
    RocCurveDisplay.from_predictions(
        y_test, y_prob, name=name, ax=ax2
    )
ax2.plot([0,1],[0,1],'k--', label='Random baseline')
ax2.set_title('ROC Curves — After SMOTE')
ax2.legend(fontsize=8)

plt.suptitle('Impact of SMOTE on Model Performance', fontsize=13)
plt.tight_layout()
plt.savefig('roc_comparison_smote.png', dpi=150)
plt.show()
print("Saved roc_comparison_smote.png")

In [ ]:

print("Building quarterly networks...")

# Make sure quarter column exists
df_exploded['quarter'] = df_exploded['created'].dt.quarter
df_with_tickers_sorted['quarter'] = pd.to_datetime(
    df_with_tickers_sorted['created']
).dt.quarter

# Store centrality per quarter per ticker
quarterly_betweenness = {}

for q in [1, 2, 3, 4]:
    print(f"\nBuilding Q{q} network...")
    
    # Use only posts from THIS quarter and earlier
    # This prevents future information leaking into past centrality
    q_posts = df_with_tickers_sorted[
        df_with_tickers_sorted['quarter'] <= q
    ].copy()
    
    # Only use multi-ticker posts for edges
    q_multi = q_posts[q_posts['num_tickers_mentioned'] > 1].copy()
    
    # Build graph
    G_q = nx.Graph()
    
    for _, row in q_multi.iterrows():
        tickers = row['tickers']
        valid = [t for t in tickers if t in valid_tickers]
        
        for i in range(len(valid)):
            for j in range(i+1, len(valid)):
                a, b = valid[i], valid[j]
                if G_q.has_edge(a, b):
                    G_q[a][b]['weight'] += 1
                else:
                    G_q.add_edge(a, b, weight=1)
    
    print(f"  Q{q} graph: {G_q.number_of_nodes()} nodes, {G_q.number_of_edges()} edges")
    
    # Compute betweenness centrality
    if G_q.number_of_edges() > 0:
        bc = nx.betweenness_centrality(G_q, normalized=True)
        quarterly_betweenness[q] = bc
    else:
        quarterly_betweenness[q] = {}
    
    print(f"  Q{q} centrality computed")

print("\nAll quarterly networks done")

In [ ]:
# Map each post to its quarter's betweenness centrality
def get_dynamic_betweenness(row):
    q = row['quarter']
    ticker = row['ticker']
    return quarterly_betweenness.get(q, {}).get(ticker, 0)

print("Mapping dynamic centrality to posts...")
df_exploded['dynamic_betweenness'] = df_exploded.apply(
    get_dynamic_betweenness, axis=1
)
print("Done")
print(df_exploded['dynamic_betweenness'].describe())

In [ ]:
# Add dynamic_betweenness to feature set
FEATURES_V2 = FEATURES + ['dynamic_betweenness']

df_model_v2 = df_exploded.dropna(subset=FEATURES_V2 + ['surge_label']).copy()
df_model_v2 = df_model_v2.sort_values('created').reset_index(drop=True)

X_v2 = df_model_v2[FEATURES_V2]
y_v2 = df_model_v2['surge_label']

# Same temporal split
split_idx_v2 = int(len(df_model_v2) * 0.8)
X_train_v2 = X_v2.iloc[:split_idx_v2]
X_test_v2  = X_v2.iloc[split_idx_v2:]
y_train_v2 = y_v2.iloc[:split_idx_v2]
y_test_v2  = y_v2.iloc[split_idx_v2:]

# Scale
scaler_v2 = StandardScaler()
X_train_v2_sc = scaler_v2.fit_transform(X_train_v2)
X_test_v2_sc  = scaler_v2.transform(X_test_v2)

# Apply SMOTE
X_train_v2_sm, y_train_v2_sm = smote.fit_resample(X_train_v2_sc, y_train_v2)

print(f"Training samples: {len(X_train_v2_sm)}")
print(f"Test samples: {len(X_test_v2)}")

In [ ]:
# Retrain best model (Gradient Boosting) with new feature
from sklearn.ensemble import GradientBoostingClassifier

gb_v2 = GradientBoostingClassifier(n_estimators=100, random_state=42)
print("Training Gradient Boosting with dynamic betweenness...")
gb_v2.fit(X_train_v2_sm, y_train_v2_sm)

y_pred_v2 = gb_v2.predict(X_test_v2_sc)
y_prob_v2 = gb_v2.predict_proba(X_test_v2_sc)[:, 1]

auc_v2  = roc_auc_score(y_test_v2, y_prob_v2)
f1_v2   = f1_score(y_test_v2, y_pred_v2, zero_division=0)
rec_v2  = recall_score(y_test_v2, y_pred_v2, zero_division=0)
prec_v2 = precision_score(y_test_v2, y_pred_v2, zero_division=0)

print(f"\n--- Gradient Boosting with Dynamic Betweenness ---")
print(f"AUC-ROC:   {auc_v2:.4f}  (before: 0.7363)")
print(f"F1:        {f1_v2:.4f}  (before: 0.2715)")
print(f"Precision: {prec_v2:.4f}  (before: 0.2246)")
print(f"Recall:    {rec_v2:.4f}  (before: 0.3433)")

In [ ]:
# Retrain all three classifiers on the final feature set (V2 + SMOTE)
# so Table 5 is a true apples-to-apples model comparison.

models_v2 = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=42
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ),
    'Gradient Boosting': GradientBoostingClassifier(
        n_estimators=100,
        random_state=42
    )
}

results_v2 = []
trained_models_v2 = {}

for name, model in models_v2.items():
    print(f"Training {name} on V2 features + SMOTE...")
    
    model.fit(X_train_v2_sm, y_train_v2_sm)
    
    y_pred = model.predict(X_test_v2_sc)
    y_prob = model.predict_proba(X_test_v2_sc)[:, 1]
    
    auc  = roc_auc_score(y_test_v2, y_prob)
    f1   = f1_score(y_test_v2, y_pred, zero_division=0)
    prec = precision_score(y_test_v2, y_pred, zero_division=0)
    rec  = recall_score(y_test_v2, y_pred, zero_division=0)
    
    results_v2.append({
        'Model':     name,
        'AUC-ROC':   round(auc,  4),
        'F1':        round(f1,   4),
        'Precision': round(prec, 4),
        'Recall':    round(rec,  4),
    })
    
    trained_models_v2[name] = (model, y_prob, y_pred)
    print(f"  AUC-ROC: {auc:.4f}  F1: {f1:.4f}  Precision: {prec:.4f}  Recall: {rec:.4f}")

print("\n--- Final Model Comparison (V2 features + SMOTE) ---")
results_v2_df = pd.DataFrame(results_v2)
print(results_v2_df.to_string(index=False))

# Save for the report
results_v2_df.to_csv('model_results_v2.csv', index=False)
print("\nSaved model_results_v2.csv")

In [ ]:
# Check where dynamic_betweenness ranks in feature importance
importances_v2 = pd.Series(
    gb_v2.feature_importances_,
    index=FEATURES_V2
).sort_values(ascending=False)

print("\nFeature importance with dynamic betweenness:")
print(importances_v2.to_string())

# Plot
fig, ax = plt.subplots(figsize=(8, 5))
importances_v2.sort_values(ascending=True).plot(
    kind='barh', ax=ax, color='steelblue', alpha=0.85
)
ax.set_title('Feature Importance — Gradient Boosting\n(with Dynamic Betweenness)', fontsize=12)
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance_v2.png', dpi=150)
plt.show()
print("Saved feature_importance_v2.png")

In [ ]:
# Final comparison table of all versions
print("=" * 60)
print("FULL COMPARISON")
print("=" * 60)
print("\nBest model (Gradient Boosting) across versions:")
print(f"{'Version':<35} {'AUC':>6} {'F1':>6} {'Recall':>8}")
print("-" * 60)
print(f"{'Original (no SMOTE)':<35} {'0.7478':>6} {'0.0000':>6} {'0.0000':>8}")
print(f"{'+ SMOTE':<35} {'0.7363':>6} {'0.2715':>6} {'0.3433':>8}")
print(f"{'+ SMOTE + Dynamic Betweenness':<35} {auc_v2:>6.4f} {f1_v2:>6.4f} {rec_v2:>8.4f}")

In [ ]:
print("Feature importance ranking:")
print(importances_v2.to_string())

In [ ]:
# Save final results
final_comparison = pd.DataFrame([
    {'Version': 'Original (no SMOTE)',
     'AUC': 0.7478, 'F1': 0.0000, 'Recall': 0.0000},
    {'Version': '+ SMOTE',
     'AUC': 0.7363, 'F1': 0.2715, 'Recall': 0.3433},
    {'Version': '+ SMOTE + Dynamic Betweenness',
     'AUC': auc_v2, 'F1': f1_v2, 'Recall': rec_v2},
])
final_comparison.to_csv('final_comparison.csv', index=False)

# Save the final model data
df_exploded.to_csv('final_data_with_network.csv', index=False)

print("Saved:")
print("  final_comparison.csv")
print("  final_data_with_network.csv")
print("  feature_importance_v2.png")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

versions = [
    'Original\n(no SMOTE)',
    '+ SMOTE',
    '+ SMOTE +\nDynamic Betweenness'
]
aucs    = [0.7478, 0.7363, auc_v2]
f1s     = [0.0000, 0.2715, f1_v2]
recalls = [0.0000, 0.3433, rec_v2]

x = np.arange(len(versions))
w = 0.25

bars1 = ax.bar(x - w,   aucs,    w, label='AUC-ROC',   color='steelblue', alpha=0.85)
bars2 = ax.bar(x,       f1s,     w, label='F1',        color='darkorange', alpha=0.85)
bars3 = ax.bar(x + w,   recalls, w, label='Recall',    color='green',      alpha=0.85)

# Add value labels on bars
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        if height > 0.01:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                height + 0.01,
                f'{height:.3f}',
                ha='center', va='bottom', fontsize=8
            )

ax.set_xticks(x)
ax.set_xticklabels(versions, fontsize=10)
ax.set_ylim(0, 1.0)
ax.set_ylabel('Score')
ax.set_title(
    'Model Improvement Across Iterations\nGradient Boosting — r/stocks 2021',
    fontsize=12
)
ax.legend(fontsize=9)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.4, label='Random baseline')
plt.tight_layout()
plt.savefig('improvement_progression.png', dpi=150)
plt.show()
print("Saved improvement_progression.png")